# Econ 3916 PS2 Warm-up — the code PS2 asks you to write

**Not graded.** Practice for Problem Set 2, on different data. Each exercise
shows you working code (*Watch*), asks you to write a similar piece yourself
(*Your turn*), then checks it. The exercises follow PS2's order: a world with a known answer, who answers, the bootstrap, coverage, then power.

- **Turn AI code completion off** (Colab: *Settings › AI assistance*), as PS2
  Phases 1–3 require. You may still click **Explain error**.
- The *Watch* cells use restaurant diners and the *Your turn* cells use gym members. The two
  differ **on purpose**: you write each piece yourself instead of copying it.
- Run the cells in order: later exercises use what earlier ones made.
- If a check fails, read its message, fix your cell, and run both again.
- Stuck for more than five minutes? Look back at the *Watch* cell above it.

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.power import TTestIndPower

---
## 1. A world with a known answer (PS2 Phase 1.1)

You simulate so that the truth is on file. `rng.lognormal(mean, sigma, size)` takes the mean and sd of the **log**, not of the values. A z-score is `(x - x.mean()) / x.std()`. A yes/no that is True with probability p is `rng.random(n) < p`. A five-point score is the latent number rounded and clipped: `np.clip(np.rint(s), 1, 5)`.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(7)
n_diners = 20_000
nights_out = rng.lognormal(mean=1.0, sigma=0.5, size=n_diners)    # mean and sd of the LOG of nights out
nights_z = (nights_out - nights_out.mean()) / nights_out.std()
has_kids = rng.random(n_diners) < 0.30                          # True for about 30% of diners
diner_latent = 3.0 + 0.40 * nights_z - 0.50 * has_kids + rng.normal(0, 0.8, size=n_diners)
stars = np.clip(np.rint(diner_latent), 1, 5)                     # a five-star scale: whole numbers 1 to 5

print(f"share with kids {has_kids.mean():.3f} | mean stars {stars.mean():.3f} | values {np.unique(stars)}")

**Your turn:** for 30,000 gym members, draw `visits` as lognormal with mean 2.0 and sigma 0.7 on the log scale, z-score it into `visits_z`, make `injured` True for 15% of members, build `latent = 3.2 + 0.30 * visits_z - 0.40 * injured + Normal(0, 0.7)`, and turn it into a 1–5 `rating`.

In [ ]:
rng = np.random.default_rng(11)
n = 30_000
# YOUR CODE HERE
visits = visits_z = injured = latent = rating = None

In [ ]:
assert rating is not None and len(rating) == 30_000, "rating needs one value per member"
assert abs(visits_z.mean()) < 1e-9 and abs(visits_z.std() - 1) < 1e-9, "z-score: subtract the mean, divide by the sd"
assert 0.14 < injured.mean() < 0.16, "injured should be True for about 15% of members"
assert set(np.unique(rating)) <= {1, 2, 3, 4, 5}, "rating must be whole numbers from 1 to 5: round, then clip"
assert rating.std() > 0.5, "rating barely varies: did the Normal(0, 0.7) noise go into latent?"
true_mean = rating.mean()
print(f"true mean rating {true_mean:.3f}: the number you keep on file")

---
## 2. Who answers (PS2 Phase 1.2)

When the chance of answering depends on the answer itself, the respondents' mean is not the population's. A probability from a score is the logistic `1 / (1 + np.exp(-z))`. Each person answers if a uniform draw falls below their probability; that gives a True/False mask, and `x[mask]` keeps the people who answered.

**Watch:** run this and read it.

In [ ]:
def leaves_review(latent, a, b, rng):
    """True for each diner who leaves a review; happier diners review more often."""
    p = 1 / (1 + np.exp(-(a + b * (latent - 3.0))))
    return rng.random(len(latent)) < p

reviewed = leaves_review(diner_latent, a=-1.5, b=1.0, rng=rng)
print(f"review rate {reviewed.mean():.1%} | all diners {stars.mean():.3f} | reviewers only {stars[reviewed].mean():.3f}")

**Your turn:** write `responds(latent, a, b, rng)` the same way, centred at 3.2 instead of 3.0, and run it on the gym members with `a = -2.0` and `b = 1.5`. Store the mask in `responded` and the respondents' mean rating in `survey_mean`.

In [ ]:
# YOUR CODE HERE
def responds(latent, a, b, rng):
    pass

responded = survey_mean = None

In [ ]:
assert responded is not None and responded.dtype == bool and len(responded) == len(rating), \
    "responded should be one True/False per member"
assert 0.10 < responded.mean() < 0.40, f"response rate {responded.mean():.1%} looks off: check the sign inside np.exp"
assert survey_mean > true_mean + 0.2, "happier members answer more, so the survey mean should sit well ABOVE the truth"
print(f"response rate {responded.mean():.1%} | truth {true_mean:.3f} | survey {survey_mean:.3f} | gap {survey_mean - true_mean:+.3f}")

---
## 3. The bootstrap, by hand and then with scipy (PS2 Phase 2.1)

A bootstrap resamples the data **with replacement** and recomputes the statistic each time; the spread of those statistics is your uncertainty. Write the loop yourself first, then check it against `stats.bootstrap`, which takes a tuple of samples, `(x,)`. **What the interval is about:** it is a range for the *average* rating, not a range for what individual members say.

**Watch:** run this and read it.

In [ ]:
reviews = stars[reviewed]
boot = np.array([rng.choice(reviews, size=len(reviews), replace=True).mean() for _ in range(2_000)])
lo_d, hi_d = np.percentile(boot, [2.5, 97.5])
res = stats.bootstrap((reviews,), np.mean, n_resamples=2_000, method="percentile", random_state=1)

print(f"by hand [{lo_d:.3f}, {hi_d:.3f}] | scipy [{res.confidence_interval.low:.3f}, {res.confidence_interval.high:.3f}]")
print(f"the interval is for the reviewers' average; it says nothing about one diner's rating (1 to 5)")

**Your turn:** bootstrap the mean of the respondents' `rating` by hand (2,000 resamples) into `boot_means`, take the 95% percentile interval as `lo, hi`, then get scipy's **BCa** interval as `bca` (`method="BCa"`, and keep the result's `.confidence_interval`).

In [ ]:
sample = rating[responded]
# YOUR CODE HERE
boot_means = None
lo = hi = None
bca = None

In [ ]:
assert boot_means is not None and len(boot_means) == 2_000, "one mean per resample, 2,000 of them"
assert lo < sample.mean() < hi, "the interval should contain the sample mean"
assert bca is not None and abs(bca.low - lo) < 0.03 and abs(bca.high - hi) < 0.03, \
    "by hand and BCa should agree to a few hundredths here"
print(f"by hand [{lo:.3f}, {hi:.3f}] | BCa [{bca.low:.3f}, {bca.high:.3f}] | truth {true_mean:.3f}")
print("Is the truth inside? A tight interval around the wrong number is still wrong.")

---
## 4. Coverage, with its Monte Carlo SE (PS2 Phase 2.2)

"95%" is a claim about repeated samples, so check it: draw many samples from a population you know, build the interval each time, and count how often it contains the truth. That share is itself an estimate; its standard error is `np.sqrt(p * (1 - p) / n_reps)`. A gap smaller than about two of those is not a finding.

**Watch:** run this and read it.

In [ ]:
hits, n_reps = 0, 400
for _ in range(n_reps):
    s = rng.choice(stars, size=200, replace=False)            # one survey of 200 diners; everyone answers
    se = s.std(ddof=1) / np.sqrt(len(s))
    hits += (s.mean() - 1.96 * se) <= stars.mean() <= (s.mean() + 1.96 * se)
cov = hits / n_reps
print(f"coverage {cov:.3f} ± {np.sqrt(cov * (1 - cov) / n_reps):.3f} (Monte Carlo SE)")

**Your turn:** now the survey with non-response. Over 400 repetitions: draw 2,000 members without replacement, keep the ones `responds(latent[idx], a=-2.0, b=1.5, rng=rng)` says answer, and build the same 95% interval for their mean. Store the share of intervals containing `true_mean` in `coverage` and its Monte Carlo SE in `mc_se`.

In [ ]:
n_reps = 400
# YOUR CODE HERE
coverage = mc_se = None

In [ ]:
assert coverage is not None and 0 <= coverage <= 1, "coverage is a share between 0 and 1"
assert coverage < 0.5, "with this much selection the interval should almost never contain the truth"
assert mc_se is not None and abs(mc_se - np.sqrt(coverage * (1 - coverage) / 400)) < 1e-12, "MC SE = sqrt(p(1-p)/n_reps)"
print(f"coverage of the truth {coverage:.3f} ± {mc_se:.3f}, against the 95% the interval claims")
print("More members per survey would not help: the bias does not shrink with n.")

---
## 5. Power, before the fact (PS2 Phase 3.2)

**Power = 1 − β: the chance a test detects an effect that is really there.** α is a different number: the false-positive rate, the chance of a "significant" result when there is no effect. `TTestIndPower().solve_power(...)` takes all but one of effect size `d`, `nobs1` (n per group), `alpha` and `power`, and solves for the one you leave out. Leave out `nobs1` to plan a sample; leave out `effect_size` to find the smallest effect a fixed sample can detect. Plan before the data: never compute power from the effect you just estimated.

**Watch:** run this and read it.

In [ ]:
d_planned = 0.15                                   # the smallest effect worth acting on, chosen BEFORE any data
n_per_group = TTestIndPower().solve_power(effect_size=d_planned, alpha=0.05, power=0.80)
print(f"to detect d = {d_planned} with power 0.80 at alpha 0.05: {n_per_group:,.0f} diners per group")
print(f"in stars: d × sd = {d_planned * stars.std(ddof=1):.3f} stars")

**Your turn:** a follow-up gym survey can reach **1,500 members per group**. Leave `effect_size` out of `solve_power` to find the smallest `d_min` it detects with power 0.80 at alpha 0.05, then turn it into rating points with the rating's standard deviation, as `points_min`.

In [ ]:
# YOUR CODE HERE
d_min = points_min = None

In [ ]:
assert d_min is not None and 0.08 < d_min < 0.13, "with 1,500 per group the detectable d is about a tenth of an sd"
assert abs(points_min - d_min * rating.std(ddof=1)) < 1e-9, "points = d × the rating's standard deviation (ddof=1)"
print(f"smallest detectable change: d = {d_min:.3f}, about {points_min:.3f} rating points")
print("Power answers 'could this design find it?'; alpha answers 'how often would it cry wolf?'")

---
**Done.** A simulated world, a response mask, a bootstrap you wrote yourself, a coverage check with its error, and a power calculation made before the data: the building blocks of PS2 Phases 1–3. Now open `ps2-starter.ipynb`.